# Video File Distraction Detection

Runs the fine-tuned MobileNetV2 model against a video file already on the board (upload one via the Jupyter file browser, or `pscp` it from your laptop), sampling roughly one frame per `SAMPLE_EVERY_N_SECONDS` of footage rather than every single frame - at ~9s/inference, processing every frame of a real video would take far longer than the video itself. Produces a summary report of every alert that fired, with the video timestamp, at the end.

Uses the same `AlertHysteresis` logic (imported from `alert_loop_infer.py`) as the live-camera and HDMI-capture notebooks.

**How to stop:** Interrupt Kernel, or it stops on its own at the end of the video.

In [ ]:
import time
import numpy as np
import cv2
from tflite_runtime import interpreter as tflite
import matplotlib.pyplot as plt
%matplotlib inline
from IPython.display import clear_output

from alert_loop_infer import AlertHysteresis, LABEL_NAMES, CONFIDENCE_THRESHOLD, post_alert, sound_buzzer_alert

MODEL_PATH = "/home/xilinx/mobilenetv2_crossview_finetuned_int8.tflite"
VIDEO_PATH = "/home/xilinx/test_drive.mp4"  # change to your uploaded video's path
SAMPLE_EVERY_N_SECONDS = 1.0  # process roughly one frame per this many seconds of footage

In [ ]:
interp = tflite.Interpreter(model_path=MODEL_PATH)
interp.allocate_tensors()
inp = interp.get_input_details()[0]
out = interp.get_output_details()[0]
print("Model loaded:", MODEL_PATH)

In [ ]:
cap = cv2.VideoCapture(VIDEO_PATH)
if not cap.isOpened():
    raise RuntimeError(f"Could not open video at {VIDEO_PATH} - check the path and that the file was uploaded")

video_fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
frame_step = max(1, int(round(video_fps * SAMPLE_EVERY_N_SECONDS)))
print(f"Video FPS: {video_fps:.1f}, sampling every {frame_step} frames (~{SAMPLE_EVERY_N_SECONDS}s of footage)")

hysteresis = AlertHysteresis()
tick = 0
alert_events = []  # (video_time_s, class_name, confidence)

try:
    frame_index = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            print("End of video reached.")
            break
        frame_index += 1
        if (frame_index - 1) % frame_step != 0:
            continue

        video_time_s = frame_index / video_fps

        img = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        resized = cv2.resize(img, (224, 224), interpolation=cv2.INTER_LINEAR)
        x = resized.astype(inp['dtype'])[None, ...]

        t0 = time.time()
        interp.set_tensor(inp['index'], x)
        interp.invoke()
        y = interp.get_tensor(out['index'])[0]
        elapsed = time.time() - t0

        class_id = int(np.argmax(y))
        confidence = float(y[class_id])
        tick += 1

        fired = hysteresis.tick(class_id, confidence)
        alert_text = "ALERT ACTIVE" if hysteresis.alert else "monitoring"
        title = (
            f"video t={video_time_s:.1f}s | {LABEL_NAMES[class_id]} ({confidence:.0%}) | {elapsed:.1f}s/frame\n"
            f"distracted_run={hysteresis.distracted_run} safe_run={hysteresis.safe_run} | {alert_text}"
        )

        clear_output(wait=True)
        plt.figure(figsize=(6, 4.5))
        plt.imshow(img)
        plt.title(title, fontsize=10, color=("red" if hysteresis.alert else "black"))
        plt.axis("off")
        plt.show()

        if fired:
            alert_events.append((video_time_s, LABEL_NAMES[class_id], confidence))
            print(f"*** ALERT FIRED at video t={video_time_s:.1f}s: {LABEL_NAMES[class_id]} ({confidence:.0%}) - posting to backend ***")
            sound_buzzer_alert()
            post_alert(LABEL_NAMES[class_id], confidence)

except KeyboardInterrupt:
    print("Stopped by user.")

cap.release()
print(f"\nProcessed {tick} sampled frame(s).")

In [ ]:
print(f"Sampled frames: {tick}")
print(f"Distinct alerts fired: {len(alert_events)}")
for t, cls, conf in alert_events:
    print(f"  video t={t:.1f}s: {cls} ({conf:.0%})")